# Sahayak - Personal object recogniser on ORBIT (transfer learning, Colab)
This is the **novel part** of the project: a few-shot recogniser for a blind user's *own* objects. A DINOv2 backbone is
fine-tuned with metric learning (last 2 blocks / LoRA) on ORBIT, a dataset filmed by blind and low-vision people, then evaluated
on **users it has never seen** with unknown objects mixed in.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Run cells top to bottom.
**Honest status:** this code passes its unit tests and a synthetic smoke run, but has NOT been run on real ORBIT data yet.
The converter was written from ORBIT's documented layout. Check the dry-run output (step 5) before training.

In [ ]:
!nvidia-smi | head -12

## 1. Drive + project + packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for d in ('checkpoints', 'reports'):
    os.makedirs(f'/content/drive/MyDrive/sahayak_orbit/{d}', exist_ok=True)

In [ ]:
!rm -rf /content/sahayak2
!git clone -q https://github.com/shreyaaaah/fewshot-object-detection-for-visually-impaired-person.git /content/sahayak2
%cd /content/sahayak2
!rm -rf checkpoints reports
!ln -sfn /content/drive/MyDrive/sahayak_orbit/checkpoints checkpoints
!ln -sfn /content/drive/MyDrive/sahayak_orbit/reports reports
!pip -q install timm scikit-learn tqdm pyyaml pytest

## 2. Quick self-test (about 2 min). Everything should say passed.

In [ ]:
!python -m pytest -q -p no:cacheprovider tests/test_memory.py tests/test_train_torch.py 2>&1 | tail -3

## 3. See the ORBIT files and their sizes (nothing is downloaded yet)

In [ ]:
import requests, shutil
files = requests.get('https://api.figshare.com/v2/articles/14294597', timeout=60).json()['files']
for f in files:
    print(f"{f['size']/1e9:8.2f} GB  {f['name']}")
want = [f for f in files if '224' in f['name'] and 'unfiltered' not in f['name'].lower()]
total = sum(f['size'] for f in want)
free = shutil.disk_usage('/content').free
print(f"\nWill download {len(want)} files, {total/1e9:.1f} GB total; free disk {free/1e9:.0f} GB")
assert want, 'no *_224 files found - paste the list above to Claude'
assert total + max(f['size'] for f in want) < free - 5e9, 'not enough disk - tell Claude the numbers above'

## 4. Download ORBIT 224 px frames, keeping every 10th frame (validation first, then train)

In [ ]:
RAW = '/content/orbit_raw'
import subprocess, zipfile, re, os
os.makedirs(RAW, exist_ok=True)
EVERY = 10                      # keep every 10th frame (ORBIT is 30 fps): small disk, fast
TMP = '/content/'
byname = {f['name']: f for f in files}

def fetch(name):
    f = byname[name]; z = TMP + name
    print('downloading', name, round(f['size'] / 1e9, 1), 'GB ...')
    subprocess.run(['wget', '-q', '-c', '-O', z, f['download_url']], check=True)
    kept = total = 0
    with zipfile.ZipFile(z) as zf:
        members = [m for m in zf.infolist() if not m.is_dir()]
        pat = re.compile(r'(\d+)\.(?:jpe?g|png)$', re.I)
        assert any(pat.search(m.filename) for m in members[:50]), 'unexpected file names - paste zf.namelist()[:10] to Claude'
        for m in members:
            total += 1
            k = pat.search(m.filename)
            if k and int(k.group(1)) % EVERY != 0:
                continue
            zf.extract(m, RAW); kept += 1
    os.remove(z)
    print(f'{name}: kept {kept} of {total} frames')

fetch('validation_224.zip')    # small one first, so the folder structure can be checked

## 5. Convert to `<user>__<object>` folders. First a DRY RUN: check that users and objects look sensible
(expect many users, names like P123__bag, and both 'clean' and 'clutter' listed).

In [ ]:
!python tools/prepare_orbit.py --src /content/orbit_raw --out data/orbit_instances --dry-run

Dry run on validation only looks right? Then download the big train file:

In [ ]:
# the big one (about 32 GB download, 10-20 min). Add 'test_224.zip' to also use ORBIT's official test users.
fetch('train_224.zip')
!du -sh {RAW}

### Now convert everything (frames are already sampled)

In [ ]:
# frames were already sampled while extracting
!python tools/prepare_orbit.py --src /content/orbit_raw --out data/orbit_instances --every 1
!ls data/orbit_instances | wc -l

## 6. Baselines: no deep learning (A0), frozen ImageNet ResNet-50 (A), frozen DINOv2 (B)

In [ ]:
ROOT = 'data/orbit_instances'
EV = f'--root {ROOT} --split test --episodes 200 --k-shots 1 3 5 10 --cross-video'
!python -m src.eval.episodic {EV} --frozen-backbone hsv_hist --tag A0_hsv_hist
!python -m src.eval.episodic {EV} --frozen-backbone resnet50.a1_in1k --tag A_resnet50_frozen
!python -m src.eval.episodic {EV} --tag B_dinov2_frozen

## 7. Transfer learning, MAIN model: last 2 blocks + head, open-set prototypical loss (about 10-20 min)

In [ ]:
EPS = 1500
!python -m src.train.train --root {ROOT} --mode partial --loss open --episodes {EPS} --tag C_partial_open --amp
!python -m src.eval.episodic {EV} --ckpt checkpoints/embedder_C_partial_open.pt --tag C_partial_open

## 8. Optional comparisons (run what time allows): LoRA, and the plain prototypical loss

In [ ]:
!python -m src.train.train --root {ROOT} --mode lora --loss open --episodes {EPS} --tag D_lora_open --amp
!python -m src.eval.episodic {EV} --ckpt checkpoints/embedder_D_lora_open.pt --tag D_lora_open

In [ ]:
!python -m src.train.train --root {ROOT} --mode partial --loss proto --episodes {EPS} --tag C_partial_proto --amp
!python -m src.eval.episodic {EV} --ckpt checkpoints/embedder_C_partial_proto.pt --tag C_partial_proto

## 9. Plots and table for the report

In [ ]:
import json, glob, pathlib
import matplotlib.pyplot as plt
R = {}
for f in sorted(glob.glob('reports/episodic_*_test.json')):
    R[pathlib.Path(f).stem.replace('episodic_', '').replace('_test', '')] = json.load(open(f))['results']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
print(f"{'model':<22}{'K':>3}{'closed-set acc %':>20}{'open-set AUROC':>18}")
for name, res in R.items():
    ks = sorted(int(k) for k in res)
    for a, key, scale, ttl in ((ax[0], 'closed_acc', 100, 'Closed-set accuracy (%) vs shots K'),
                               (ax[1], 'auroc_ours_prob', 1, 'Open-set AUROC (unknown-object rejection) vs K')):
        m = [res[str(k)][key][0] * scale for k in ks]
        c = [res[str(k)][key][1] * scale for k in ks]
        a.errorbar(ks, m, yerr=c, marker='o', capsize=3, label=name)
        a.set_title(ttl); a.set_xlabel('K (examples per object)'); a.grid(alpha=.3)
    for k in ks:
        print(f"{name:<22}{k:>3}{res[str(k)]['closed_acc'][0]*100:>14.1f} +- {res[str(k)]['closed_acc'][1]*100:<4.1f}{res[str(k)]['auroc_ours_prob'][0]:>12.3f}")
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig('reports/orbit_results.png', dpi=150); plt.show()

## 10. Download

In [ ]:
!zip -qr orbit_results.zip reports
from google.colab import files
files.download('/content/sahayak2/orbit_results.zip')
files.download('/content/sahayak2/checkpoints/embedder_C_partial_open.pt')